# Scenario 2 — Minitron: Aggressive Depth Pruning (Comparison Baseline) (~45 min on 2x H200)

This notebook prunes Qwen3-8B from 36 layers down to 22 layers using **Minitron** depth pruning, then distills and evaluates the result.

This serves as the **comparison baseline** for Scenario 2. The recommended approach for aggressive compression is Puzzletron (see [`scenario2_puzzletron.ipynb`](scenario2_puzzletron.ipynb)). We run Minitron here to demonstrate why depth pruning underperforms at this compression level.

**Pipeline:** Prune → Evaluate → Distill → Evaluate

**Prerequisites:**
- Run [`00_prerequisites.ipynb`](00_prerequisites.ipynb) first to prepare the distillation dataset.
- Base model downloaded at `/workspace/models/Qwen3-8B`.

## 1. Prune (36 → 22 layers)

To match the ~78,000 MiB memory budget used in the Puzzletron comparison, we need aggressive compression. With Minitron, the most effective way to achieve large memory savings is **depth pruning** — removing entire Transformer layers. Each layer carries not only its weights but also a KV cache allocation at inference time, so dropping a layer saves both weight memory and KV cache memory. This makes depth pruning far more memory-efficient per parameter removed than width pruning alone.

**Why 22 layers?** Each Qwen3-8B layer accounts for ~3,440 MiB at the inference settings used in this guide (KV cache + attention + FFN weights; the full breakdown is computed in [`scenario2_puzzletron.ipynb`](scenario2_puzzletron.ipynb)). Dropping 14 of the 36 layers removes ~48,160 MiB, taking the baseline from 126,215 MiB down to ~78,055 MiB — within the 78,000 MiB target.

Minitron ranks the layers to decide which 14 layers to drop, keeping the 22 that contribute most to the model's output quality.

In [ ]:
!cd /opt/Model-Optimizer/examples/megatron_bridge && \
torchrun --nproc_per_node 2 prune_minitron.py \
    --pp_size 2 \
    --hf_model_name_or_path /workspace/models/Qwen3-8B \
    --prune_export_config '{"num_layers": 22}' \
    --output_hf_path /workspace/output/Qwen3-8B-Minitron-22L

## 2. Verify pruned model

In [ ]:
!ls -lh /workspace/output/Qwen3-8B-Minitron-22L/

## 3. Evaluate pruned model (before distillation)

With 14 layers removed (~39% of the model's depth), we expect a significant accuracy drop. At this level of compression, the model may be near-random on MMLU.

In [ ]:
!lm_eval --model hf \
    --model_args pretrained=/workspace/output/Qwen3-8B-Minitron-22L,dtype=bfloat16 \
    --tasks mmlu \
    --num_fewshot 5 \
    --batch_size 4

## 4. Distill

Launch TensorBoard to monitor the distillation loss in real time. Open http://localhost:6006 in your browser once the distillation cell is running.

> **Tip:** In the TensorBoard settings (top-right gear icon), check **"Reload data"** so the charts update automatically as training progresses.

In [ ]:
import subprocess

subprocess.Popen(
    ["tensorboard", "--logdir", "/workspace/output/distill_output_22L/tb_logs", "--port", "6006"]
)
print("TensorBoard started at http://localhost:6006")


Distill the 22-layer model against the full Qwen3-8B teacher. Same setup as Scenario 1: 100 iterations on [WikiText-103](https://huggingface.co/datasets/Salesforce/wikitext/tree/main/wikitext-103-v1).
> **Expected runtime: ~20-30 minutes on 2x H200.**

In [ ]:
!cd /opt/Model-Optimizer/examples/megatron_bridge && \
torchrun --nnodes 1 --nproc_per_node=2 distill.py \
    --student_hf_path /workspace/output/Qwen3-8B-Minitron-22L \
    --teacher_hf_path /workspace/models/Qwen3-8B \
    --data_paths 1.0 /workspace/datasets/tokenized_qwen3/wikitext_wikitext-103-v1_train_text \
    --output_dir /workspace/output/distill_output_22L \
    --hf_export_path /workspace/output/distilled_Qwen3-8B-Minitron-22L \
    --student_hf_model /workspace/output/Qwen3-8B-Minitron-22L \
    --seq_length 4096 \
    --tp_size 2 \
    --pp_size 1 \
    --mbs 1 \
    --gbs 4 \
    --train_iters 100 \
    --lr 0.0001 \
    --min_lr 1e-05 \
    --lr_warmup_iters 10 \
    --eval_interval 10 \
    --eval_iters 10 \
    --log_interval 1

Finally, kill tensorboard:

In [ ]:
subprocess.run(["pkill", "-f", "tensorboard"])

## 5. Evaluate distilled model

Compare with the Puzzletron result at the same memory budget (see [`scenario2_puzzletron.ipynb`](scenario2_puzzletron.ipynb)).

**Expected results on Qwen3-8B:**

| Model | Memory | MMLU (5-shot) | % of Teacher |
|---|---|---|---|
| Qwen3-8B (teacher) | 126,215 MiB | 0.7493 | 100% |
| Minitron 22L — pruned | 78,054 MiB | 0.2351 | 31.4% |
| Minitron 22L — distilled | 78,054 MiB | 0.4620 | 61.7% |
| **Puzzletron 78k — distilled** | **77,992 MiB** | **0.5613** | **74.9%** |

In [ ]:
!lm_eval --model hf \
    --model_args pretrained=/workspace/output/distilled_Qwen3-8B-Minitron-22L,dtype=bfloat16 \
    --tasks mmlu \
    --num_fewshot 5 \
    --batch_size 4